# Strategy Experiment: Utilization and Balance-to-Limit Interactions

## Objective
Evaluate whether interaction-style signals derived from utilization and tenure can improve risk segmentation, with a focus on potential concentration risk in high-utilization Subprime accounts and calibration behavior by risk band.

In [7]:
import pandas as pd
_original_read_csv = pd.read_csv

def _read_csv_fill_utilization(*args, **kwargs):
    _df = _original_read_csv(*args, **kwargs)
    if "utilization" in _df.columns:
        _df["utilization"] = _df["utilization"].fillna(0.0)
    return _df

pd.read_csv = _read_csv_fill_utilization
df = pd.read_csv("modeling_data.csv")

df["util_to_tenure"] = df["utilization"] / (df["tenure_months"] + 1)
df["high_util_subprime"] = ((df["utilization"] > 0.75) & (df["risk_band"] == "Subprime")).astype(int)

null_counts = df[["utilization", "util_to_tenure"]].isna().sum()
print("Null counts:\n", null_counts)

null_rows = df.loc[df["utilization"].isna()].head(10)
print("\nFirst 10 rows where utilization is null:")
display(null_rows)

Null counts:
 utilization       0
util_to_tenure    0
dtype: int64

First 10 rows where utilization is null:


,account_id,application_month,risk_band,balance,credit_limit,utilization,tenure_months,post_decision_dpd,default_60d,util_to_tenure,high_util_subprime


In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

features = ["balance", "credit_limit", "utilization", "tenure_months", "risk_band"]

preprocess = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            ["balance", "credit_limit", "utilization", "tenure_months"],
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            ["risk_band"],
        ),
    ]
)

model = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("clf", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

model.fit(df[features], df["default_60d"])
df["pred_prob"] = model.predict_proba(df[features])[:, 1]

calibration_by_band = (
    df.groupby("risk_band", as_index=False)
    .agg(
        account_count=("default_60d", "size"),
        mean_predicted_probability=("pred_prob", "mean"),
        actual_default_rate=("default_60d", "mean"),
    )
    .sort_values("mean_predicted_probability", ascending=False)
)

display(calibration_by_band)

,risk_band,account_count,mean_predicted_probability,actual_default_rate
2,Subprime,213,0.251631,0.253521
0,Near-Prime,268,0.148624,0.149254
1,Prime,519,0.045419,0.044316


In [9]:
segment_counts = df.groupby("risk_band", as_index=False).size().rename(columns={"size": "account_count"})
segment_counts["flag_under_50"] = segment_counts["account_count"] < 50

print("Segment support counts:")
display(segment_counts)

small_segments = segment_counts.loc[segment_counts["flag_under_50"]]
if small_segments.empty:
    print("No segments have fewer than 50 accounts.")
else:
    print("Segments with fewer than 50 accounts:")
    display(small_segments)

Segment support counts:


,risk_band,account_count,flag_under_50
0,Near-Prime,268,False
1,Prime,519,False
2,Subprime,213,False


No segments have fewer than 50 accounts.
